# GRASP + LMM (Titans) + code_mapping: Mortality Prediction on MIMIC-III

This notebook runs **GRASP with LMM backbone** and `code_mapping` on the full MIMIC-III dataset.

**Papers**:
- Ali Behrouz et al. "Titans: Learning to Memorize at Test Time." arXiv 2501.00663, 2025.
- Liantao Ma et al. "GRASP: Generic Framework for Health Status Representation Learning." AAAI 2021.

**Context**: LMM standalone + code_mapping achieved roc_auc ~0.521. This run adds GRASP clustering to test whether cross-patient knowledge further improves LMM's surprise-based representations.

**Previous results:**
- LMM standalone, no code_mapping:   roc_auc = 0.521
- GRASP+LMM, no code_mapping:        roc_auc = 0.560
- GRASP+GRU, with code_mapping:      roc_auc = 0.639 (sweep best)

### Compute Tracking
Tracks energy, GPU utilization, and CO2 emissions via `codecarbon` and `pynvml`.

## Step 1: Load the MIMIC-III Dataset

We load the MIMIC-III dataset using PyHealth's `MIMIC3Dataset` class.

**Two configurations:**
- **Local (default):** Set `dev=True` with synthetic GCS data for quick testing
- **H200 / full run:** Set `USE_LOCAL_DATA = True` below and point `LOCAL_ROOT` to your MIMIC-III CSV directory

In [ ]:
import tempfile

from pyhealth.datasets import MIMIC3Dataset

# ── Configuration ─────────────────────────────────────────
# Set USE_LOCAL_DATA = True on the H200 where MIMIC-III CSVs live
USE_LOCAL_DATA = True
LOCAL_ROOT = "/home/lolowo2"           # directory with ADMISSIONS.csv.gz, etc.
LOCAL_CACHE = "/home/lolowo2/tmp/pyhealth_cache"
# ──────────────────────────────────────────────────────────

if USE_LOCAL_DATA:
    base_dataset = MIMIC3Dataset(
        root=LOCAL_ROOT,
        tables=["DIAGNOSES_ICD", "PROCEDURES_ICD", "PRESCRIPTIONS"],
        cache_dir=LOCAL_CACHE,
        dev=False,  # full dataset
    )
else:
    base_dataset = MIMIC3Dataset(
        root="https://storage.googleapis.com/pyhealth/Synthetic_MIMIC-III",
        tables=["DIAGNOSES_ICD", "PROCEDURES_ICD", "PRESCRIPTIONS"],
        cache_dir=tempfile.TemporaryDirectory().name,
        dev=True,  # small synthetic subset
    )

base_dataset.stats()

## Step 2: Define the Mortality Prediction Task (with code_mapping)

The `MortalityPredictionMIMIC3` task with `code_mapping` enabled:
- **conditions**: ICD-9-CM → CCS-CM (~530 groups)
- **procedures**: ICD-9-PROC → CCS-PROC (~230 groups)
- **drugs**: NDC → ATC (~6,000 groups)

This collapses the vocabulary so each embedding gets trained on enough data to be meaningful.

In [ ]:
from pyhealth.tasks import MortalityPredictionMIMIC3

# Enable code_mapping to collapse granular codes into grouped vocabularies
task = MortalityPredictionMIMIC3(
    code_mapping={
        "conditions": ("ICD9CM", "CCSCM"),
        "procedures": ("ICD9PROC", "CCSPROC"),
        "drugs": ("NDC", "ATC"),
    }
)

samples = base_dataset.set_task(task)

print(f"Generated {len(samples)} samples")
print(f"\nInput schema: {samples.input_schema}")
print(f"Output schema: {samples.output_schema}")

## Step 3: Split Data and Create Loaders

We split by patient to avoid data leakage, then create batched data loaders.

In [ ]:
from pyhealth.datasets import split_by_patient, get_dataloader

train_dataset, val_dataset, test_dataset = split_by_patient(
    samples, [0.8, 0.1, 0.1]
)

train_dataloader = get_dataloader(train_dataset, batch_size=32, shuffle=True)
val_dataloader = get_dataloader(val_dataset, batch_size=32, shuffle=False)
test_dataloader = get_dataloader(test_dataset, batch_size=32, shuffle=False)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")

## Compute Tracking Setup

We log hardware specs, energy consumption, GPU power/utilization, and CO2 emissions.

Install dependencies if needed:
```bash
pip install codecarbon pynvml
```

**Metrics tracked per training run:**

| Metric | What it measures | Unit |
|---|---|---|
| Wall time | Real elapsed clock time | seconds |
| Energy consumed | Electricity drawn by GPU | kWh |
| CO2 emissions | Energy × grid carbon intensity | kg CO2eq |
| Peak / avg GPU power | How hard the chip works | Watts |
| GPU utilization | % time GPU cores are computing | % |
| GPU memory | VRAM allocated vs available | GB |
| Energy per epoch | Diminishing returns indicator | kWh |
| Energy per sample | Unit cost for deployment | Joules |
| Batch throughput | Samples processed per second | samples/s |

In [ ]:
import time
import platform
import torch

# ── Hardware Info ──────────────────────────────────────────
print("=" * 60)
print("Hardware Information")
print("=" * 60)
print(f"Platform:     {platform.system()} {platform.machine()}")
print(f"Python:       {platform.python_version()}")
print(f"PyTorch:      {torch.__version__}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU:          {gpu_name}")
    print(f"GPU Memory:   {gpu_mem_gb:.1f} GB")
    print(f"CUDA:         {torch.version.cuda}")
else:
    print("GPU:          None (CPU only)")
    print("  Note: Compute metrics will be limited without GPU")

# ── GPU Monitoring Helper ─────────────────────────────────
gpu_available = torch.cuda.is_available()
nvml_available = False

if gpu_available:
    try:
        import pynvml
        pynvml.nvmlInit()
        nvml_handle = pynvml.nvmlDeviceGetHandleByIndex(0)
        nvml_available = True
        print(f"\npynvml:       initialized (live GPU monitoring enabled)")
    except (ImportError, Exception) as e:
        print(f"\npynvml:       not available ({e})")
        print("  Install with: pip install pynvml")

# ── Carbon Tracking Helper ────────────────────────────────
codecarbon_available = False
try:
    from codecarbon import EmissionsTracker
    codecarbon_available = True
    print(f"codecarbon:   available (CO2 tracking enabled)")
except ImportError:
    print(f"codecarbon:   not available")
    print("  Install with: pip install codecarbon")


class ComputeTracker:
    """Tracks energy, GPU metrics, and timing for a training run."""

    def __init__(self, name, num_samples, num_epochs):
        self.name = name
        self.num_samples = num_samples
        self.num_epochs = num_epochs
        self.power_readings = []
        self.util_readings = []
        self.mem_readings = []

    def start(self):
        self.start_time = time.time()
        if gpu_available:
            torch.cuda.reset_peak_memory_stats()
            torch.cuda.synchronize()
        if codecarbon_available:
            self.tracker = EmissionsTracker(
                project_name=self.name,
                log_level="error",
                save_to_file=False,
            )
            self.tracker.start()
        self._sample_gpu()
        return self

    def _sample_gpu(self):
        if nvml_available:
            power = pynvml.nvmlDeviceGetPowerUsage(nvml_handle) / 1000  # mW -> W
            util = pynvml.nvmlDeviceGetUtilizationRates(nvml_handle).gpu
            mem_info = pynvml.nvmlDeviceGetMemoryInfo(nvml_handle)
            self.power_readings.append(power)
            self.util_readings.append(util)
            self.mem_readings.append(mem_info.used / 1e9)

    def sample(self):
        """Call periodically during training to collect GPU readings."""
        self._sample_gpu()

    def stop(self):
        if gpu_available:
            torch.cuda.synchronize()
        self.wall_time = time.time() - self.start_time
        self._sample_gpu()

        self.emissions_data = None
        if codecarbon_available:
            self.tracker.stop()
            self.emissions_data = self.tracker.final_emissions_data

        return self.report()

    def report(self):
        r = {"name": self.name}
        r["wall_time_s"] = self.wall_time
        r["wall_time_min"] = self.wall_time / 60
        r["num_epochs"] = self.num_epochs
        r["num_samples"] = self.num_samples

        # Throughput
        total_samples_processed = self.num_samples * self.num_epochs
        r["batch_throughput_sps"] = total_samples_processed / self.wall_time
        r["time_per_epoch_s"] = self.wall_time / max(self.num_epochs, 1)

        # Energy (from codecarbon)
        if self.emissions_data:
            r["energy_kwh"] = self.emissions_data.energy_consumed
            r["co2_kg"] = self.emissions_data.emissions
            r["energy_per_epoch_kwh"] = r["energy_kwh"] / max(self.num_epochs, 1)
            r["energy_per_sample_j"] = (r["energy_kwh"] * 3.6e6) / max(total_samples_processed, 1)
        else:
            r["energy_kwh"] = None
            r["co2_kg"] = None

        # GPU metrics (from pynvml)
        if self.power_readings:
            r["peak_power_w"] = max(self.power_readings)
            r["avg_power_w"] = sum(self.power_readings) / len(self.power_readings)
            r["avg_gpu_util_pct"] = sum(self.util_readings) / len(self.util_readings)
            r["peak_gpu_mem_gb"] = max(self.mem_readings)
        elif gpu_available:
            r["peak_gpu_mem_gb"] = torch.cuda.max_memory_allocated() / 1e9

        return r


def print_report(r):
    print(f"\n{'=' * 50}")
    print(f"Compute Report: {r['name']}")
    print(f"{'=' * 50}")
    print(f"Wall time:          {r['wall_time_s']:.1f}s ({r['wall_time_min']:.2f} min)")
    print(f"Epochs:             {r['num_epochs']}")
    print(f"Time per epoch:     {r['time_per_epoch_s']:.2f}s")
    print(f"Throughput:         {r['batch_throughput_sps']:.1f} samples/s")

    if r.get("energy_kwh") is not None:
        print(f"\nEnergy consumed:    {r['energy_kwh']:.6f} kWh")
        print(f"CO2 emissions:      {r['co2_kg']:.6f} kg CO2eq")
        print(f"Energy per epoch:   {r['energy_per_epoch_kwh']:.6f} kWh")
        print(f"Energy per sample:  {r['energy_per_sample_j']:.4f} J")

    if r.get("peak_power_w"):
        print(f"\nPeak GPU power:     {r['peak_power_w']:.0f} W")
        print(f"Avg GPU power:      {r['avg_power_w']:.0f} W")
        print(f"Avg GPU utilization: {r['avg_gpu_util_pct']:.0f}%")

    if r.get("peak_gpu_mem_gb"):
        total_gb = gpu_mem_gb if gpu_available else 0
        used = r["peak_gpu_mem_gb"]
        print(f"Peak GPU memory:    {used:.2f} GB / {total_gb:.0f} GB ({used/max(total_gb,1)*100:.0f}%)")

    print(f"{'=' * 50}")


print("\nComputeTracker ready.")

## Step 4: GRASP + LMM + code_mapping

GRASP wraps LMM with patient clustering (k-means), cluster refinement (GCN), and a learned gate that blends each patient's own LMM representation with cross-patient knowledge.

- **LMM**: "What should I remember from *this* patient's history?"
- **GRASP**: "What can I learn from *similar* patients?"

In [ ]:
from pyhealth.models import GRASP

NUM_EPOCHS = 50

grasp_lmm_model = GRASP(
    dataset=samples,
    embedding_dim=32,
    hidden_dim=32,
    cluster_num=8,
    block="LMM",
)

print(f"GRASP+LMM model: {sum(p.numel() for p in grasp_lmm_model.parameters()):,} parameters")

In [ ]:
from pyhealth.trainer import Trainer

grasp_lmm_trainer = Trainer(
    model=grasp_lmm_model,
    metrics=["roc_auc", "pr_auc", "accuracy", "f1"],
)

grasp_tracker = ComputeTracker(
    "GRASP+LMM+codemap", len(train_dataset), NUM_EPOCHS,
).start()

grasp_lmm_trainer.train(
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    epochs=NUM_EPOCHS,
    monitor="roc_auc",
    optimizer_params={"lr": 5e-4, "weight_decay": 1e-4},
)

grasp_compute = grasp_tracker.stop()
print_report(grasp_compute)

In [ ]:
grasp_lmm_results = grasp_lmm_trainer.evaluate(test_dataloader)

print("=" * 50)
print("GRASP + LMM + code_mapping — Test Set Performance")
print("=" * 50)
for metric, value in grasp_lmm_results.items():
    print(f"{metric}: {value:.4f}")

## Step 5: Results

GRASP+LMM+code_mapping performance and compute cost.

In [ ]:
grasp_lmm_results = grasp_lmm_trainer.evaluate(test_dataloader)

print("=" * 60)
print("GRASP + LMM + code_mapping — Test Set Performance")
print("=" * 60)
for metric, value in grasp_lmm_results.items():
    print(f"  {metric}: {value:.4f}")

print("\n--- Reference ---")
print("  LMM standalone, no code_mapping:   roc_auc = 0.521")
print("  GRASP+LMM, no code_mapping:        roc_auc = 0.560")
print("  GRASP+GRU, with code_mapping:      roc_auc = 0.639")

## Step 7: Extract Patient Embeddings

Both models produce patient embeddings that can be used for downstream tasks like patient similarity search or cohort discovery.

In [ ]:
import torch

grasp_lmm_model.eval()
test_batch = next(iter(test_dataloader))
test_batch["embed"] = True

with torch.no_grad():
    output = grasp_lmm_model(**test_batch)

print(f"Embedding shape: {output['embed'].shape}")
print(f"  - Batch size: {output['embed'].shape[0]}")
print(f"  - Embedding dim: {output['embed'].shape[1]}")

print("\nSample Predictions:")
predictions = output["y_prob"].cpu().numpy()
true_labels = output["y_true"].cpu().numpy()

for i in range(min(5, len(predictions))):
    pred = predictions[i][0]
    true = int(true_labels[i][0])
    label = "Mortality" if pred > 0.5 else "Survival"
    print(f"  Patient {i + 1}: Predicted={pred:.3f}, True={true}, -> {label}")

## Step 8: Compute Cost Summary for Policy

This section translates raw compute metrics into policy-relevant context. The goal is to answer: **what is the real environmental cost of this ML research?**

In [ ]:
from datetime import datetime

total_energy = grasp_compute.get("energy_kwh") or 0
total_co2 = grasp_compute.get("co2_kg") or 0
total_time = grasp_compute["wall_time_s"]

print("=" * 60)
print("ENVIRONMENTAL IMPACT — POLICY SUMMARY")
print("=" * 60)
print(f"\n--- GRASP+LMM+code_mapping footprint ---")
print(f"Wall time:            {total_time:.0f}s ({total_time/60:.1f} min)")

if total_energy > 0:
    lightbulb_min = (total_energy / 0.01) * 60
    home_fraction = total_energy / 30 * 100
    miles_driven = total_co2 / 0.4

    print(f"Energy consumed:      {total_energy:.6f} kWh")
    print(f"CO2 emissions:        {total_co2:.6f} kg CO2eq")
    print(f"\n--- In everyday terms ---")
    print(f"10W LED bulb for:     {lightbulb_min:.0f} minutes")
    print(f"US home daily usage:  {home_fraction:.4f}%")
    print(f"Equivalent driving:   {miles_driven:.3f} miles")

    FULL_SWEEP_CONFIGS = 108
    print(f"\n--- Extrapolated: full sweep ({FULL_SWEEP_CONFIGS} configs) ---")
    print(f"Energy:   {total_energy * FULL_SWEEP_CONFIGS:.2f} kWh")
    print(f"CO2:      {total_co2 * FULL_SWEEP_CONFIGS:.2f} kg CO2eq")
    print(f"Driving:  {total_co2 * FULL_SWEEP_CONFIGS / 0.4:.0f} miles")

if grasp_compute.get("avg_gpu_util_pct") is not None:
    util = grasp_compute["avg_gpu_util_pct"]
    mem = grasp_compute.get("peak_gpu_mem_gb", 0)
    total_gb = gpu_mem_gb if gpu_available else 80
    print(f"\n--- GPU efficiency ---")
    print(f"Avg GPU utilization: {util:.0f}%")
    print(f"Memory:              {mem:.1f} GB / {total_gb:.0f} GB ({mem/total_gb*100:.0f}%)")
    if mem < total_gb * 0.1:
        print(f"  -> Could run on < {max(4, mem*2):.0f} GB GPU (consumer card)")

print("=" * 60)